In [1]:
import os, shutil

base_path = "/kaggle/input/globular-clusters5"
work_path = "/kaggle/working/globular-clusters5-fixed"

os.makedirs(os.path.join(work_path, "clusters"), exist_ok=True)

# Copy all images into the "clusters" subfolder
for file in os.listdir(base_path):
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        shutil.copy(os.path.join(base_path, file),
                    os.path.join(work_path, "clusters", file))

print("✅ All images moved into /clusters/ subfolder")


✅ All images moved into /clusters/ subfolder


In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Transform pipeline
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # normalize to [-1,1]
])

dataset = datasets.ImageFolder(root=work_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Loaded {len(dataset)} images for training.")

Loaded 224 images for training.


In [3]:
import torch.nn as nn

# Generator
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)


In [4]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

netG = Generator().to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, 100, 1, 1, device=device)

for epoch in range(10):  # adjust as needed
    for i, (data, _) in enumerate(dataloader):
        real = data.to(device)
        b_size = real.size(0)
        label = torch.full((b_size,), 1., dtype=torch.float, device=device)

        # Train D with real
        netD.zero_grad()
        output = netD(real)
        errD_real = criterion(output, label)
        errD_real.backward()

        # Train D with fake
        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(0.)
        output = netD(fake.detach())
        errD_fake = criterion(output, label)
        errD_fake.backward()
        optimizerD.step()

        # Train G
        netG.zero_grad()
        label.fill_(1.)
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        optimizerG.step()

    print(f"Epoch [{epoch+1}/10] Loss_D: {errD_real+errD_fake:.4f} Loss_G: {errG:.4f}")


ValueError: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([1600])) is deprecated. Please ensure they have the same size.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # input is (nc) x 128 x 128
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),   # 64x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False), # 32x32
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False), # 16x16
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False), # 8x8
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, ndf*16, 4, 2, 1, bias=False), # 4x4
            nn.BatchNorm2d(ndf*16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*16, 1, 4, 1, 0, bias=False),     # 1x1
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)


In [ ]:
netD = Discriminator()
x = torch.randn(64, 3, 128, 128)  # batch of 64 RGB images
out = netD(x)
print(out.shape)  # should be [64]

In [ ]:
# 1. Imports
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# 2. Paths
data_path = "/kaggle/working/globular-clusters5-fixed/clusters"  # your prepared folder
batch_size = 64
image_size = 128

# 3. Dataset
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)  # normalize to [-1,1]
])

dataset = datasets.ImageFolder(root=os.path.dirname(data_path), transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Loaded {len(dataset)} images")

# 4. Generator
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64, nc=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf*16, 4, 1, 0, bias=False),  # 1x1 -> 4x4
            nn.BatchNorm2d(ngf*16),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*16, ngf*8, 4, 2, 1, bias=False), # 4x4 -> 8x8
            nn.BatchNorm2d(ngf*8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False), # 8x8 -> 16x16
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False), # 16x16 -> 32x32
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),    # 32x32 -> 64x64
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),       # 64x64 -> 128x128
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

# 5. Discriminator
class Discriminator(nn.Module):
    def __init__(self, nc=3, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),  # 128 -> 64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False), # 64 -> 32
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False), # 32 -> 16
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False), # 16 -> 8
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, ndf*16, 4, 2, 1, bias=False), # 8 -> 4
            nn.BatchNorm2d(ndf*16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*16, 1, 4, 1, 0, bias=False),      # 4 -> 1
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input).view(-1)

# 6. Initialize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
netG = Generator().to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# 7. Training (minimal example: 1 epoch)
for epoch in range(1):
    for i, (data, _) in enumerate(dataloader):
        real = data.to(device)
        b_size = real.size(0)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # Train Generator
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        if i % 10 == 0:
            print(f"Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")

# 8. Generate samples
with torch.no_grad():
    fake = netG(fixed_noise).detach().cpu()

grid = vutils.make_grid(fake, padding=2, normalize=True)
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Generated Globular Clusters")
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 5        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 5        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 5        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 5        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 5        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 5        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors for monitoring progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Parameters
max_epochs = 1000          # large number, loop will stop manually
display_interval = 50      # show output every 50 batches
target_quality = 0.6       # optional dummy threshold to stop based on loss (you can ignore if you just want fixed iterations)

epoch = 0
while epoch < max_epochs:
    epoch += 1
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

    # optional: stopping condition based on visual inspection
    # if your images look good, you can manually interrupt the notebook


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 20        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch

# Fixed latent vectors to monitor progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters for faster progress
display_interval = 5  # show output every 5 batches
num_epochs = 100        # keep small if testing
gen_lr = 0.0004       # slightly faster learning for generator

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Training loop
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
num_epochs = 100
batch_size = 128
lr = 0.0002               # same for both G and D
beta1 = 0.5
beta2 = 0.999
display_interval = 100    # batches between console logging

# Assume your models exist: netG, netD
# Example:
# netG = Generator().to(device)
# netD = Discriminator().to(device)

# Loss function
criterion = nn.BCELoss()

# Optimizers
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # -----------------
        # Train Discriminator
        # -----------------
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, 100, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real).view(-1)
            lossD_real = criterion(output_real, label_real)

            fake = netG(noise)
            output_fake = netD(fake.detach()).view(-1)
            lossD_fake = criterion(output_fake, label_fake)

            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # -----------------
        # Train Generator
        # -----------------
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake).view(-1)
            lossG = criterion(output, label_real)

        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        # -----------------
        # Logging
        # -----------------
        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

print("Training complete.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
num_epochs = 100
batch_size = 128
lr = 0.0002               # same for both G and D
beta1 = 0.5
beta2 = 0.999
display_interval = 100    # batches between console logging

# Assume your models exist: netG, netD
# Example:
# netG = Generator().to(device)
# netD = Discriminator().to(device)

# Loss function (logits version, no Sigmoid in netD)
criterion = nn.BCEWithLogitsLoss()

# Optimizers
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # -----------------
        # Train Discriminator
        # -----------------
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, 100, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real).view(-1)
            lossD_real = criterion(output_real, label_real)

            fake = netG(noise)
            output_fake = netD(fake.detach()).view(-1)
            lossD_fake = criterion(output_fake, label_fake)

            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # -----------------
        # Train Generator
        # -----------------
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake).view(-1)
            lossG = criterion(output, label_real)

        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        # -----------------
        # Logging
        # -----------------
        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

print("Training complete.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# -----------------------------
# Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed noise for monitoring progress
fixed_noise = torch.randn(64, 100, 1, 1, device=device)

# Hyperparameters
num_epochs = 100
batch_size = 128
lr = 0.0002               # same for both G and D
beta1 = 0.5
beta2 = 0.999
display_interval = 100    # batches between console logging

# Assume your models exist: netG, netD
# Example:
# netG = Generator().to(device)
# netD = Discriminator().to(device)

# Loss function (safe for AMP, no Sigmoid in netD)
criterion = nn.BCEWithLogitsLoss()

# Optimizers
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # -----------------
        # Train Discriminator
        # -----------------
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, 100, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real).view(-1)
            lossD_real = criterion(output_real, label_real)

            fake = netG(noise)
            output_fake = netD(fake.detach()).view(-1)
            lossD_fake = criterion(output_fake, label_fake)

            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # -----------------
        # Train Generator
        # -----------------
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake).view(-1)
            lossG = criterion(output, label_real)

        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        # -----------------
        # Logging
        # -----------------
        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # -----------------
    # Show results at the end of each epoch
    # -----------------
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)

    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1}")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()

print("Training complete.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os

# -----------------------------
# Device and Hyperparameters
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fixed_noise = torch.randn(64, 100, 1, 1, device=device)  # for consistent visual tracking

num_epochs = 100
batch_size = 64           # smaller batch for Kaggle stability
lr = 0.0002
beta1 = 0.5
beta2 = 0.999
display_interval = 100    # batches between logging

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Models, Loss, Optimizers
# -----------------------------
# Define your Generator and Discriminator here
# Example:
# netG = Generator().to(device)
# netD = Discriminator().to(device)

criterion = nn.BCEWithLogitsLoss()  # stable with AMP, netD outputs raw logits
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, beta2))
scaler = torch.cuda.amp.GradScaler()  # mixed precision

# -----------------------------
# Checkpoint Functions
# -----------------------------
def save_checkpoint(epoch, netG, netD, optimizerG, optimizerD, scaler):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

def load_checkpoint(netG, netD, optimizerG, optimizerD, scaler, device):
    ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
    if ckpt_files:
        latest_ckpt = os.path.join(checkpoint_dir, sorted(ckpt_files)[-1])
        checkpoint = torch.load(latest_ckpt, map_location=device)
        netG.load_state_dict(checkpoint["netG"])
        netD.load_state_dict(checkpoint["netD"])
        optimizerG.load_state_dict(checkpoint["optimizerG"])
        optimizerD.load_state_dict(checkpoint["optimizerD"])
        scaler.load_state_dict(checkpoint["scaler"])
        start_epoch = checkpoint["epoch"] + 1
        print(f"Resuming from epoch {start_epoch}")
        return start_epoch
    return 0

# -----------------------------
# Load checkpoint if exists
# -----------------------------
start_epoch = load_checkpoint(netG, netD, optimizerG, optimizerD, scaler, device)

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # -----------------
        # Train Discriminator
        # -----------------
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, 100, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real).view(-1)
            lossD_real = criterion(output_real, label_real)

            fake = netG(noise)
            output_fake = netD(fake.detach()).view(-1)
            lossD_fake = criterion(output_fake, label_fake)

            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # -----------------
        # Train Generator
        # -----------------
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake).view(-1)
            lossG = criterion(output, label_real)

        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        # -----------------
        # Batch Logging
        # -----------------
        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # -----------------
    # End of epoch: show results
    # -----------------
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)

    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()  # release memory

    # -----------------
    # Save checkpoint
    # -----------------
    save_checkpoint(epoch, netG, netD, optimizerG, optimizerD, scaler)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 64        # depends on your dataset
nz = 100               # latent vector size
display_interval = 100
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1, 1] range
])

# Example dataset (replace with your dataset)
# dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator and Discriminator (DCGAN)
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False),
            nn.Tanh()  # output in [-1, 1]
        )

    def forward(self, x):
        return self.main(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.main(x).view(-1)

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, Loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint functions
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

def load_checkpoint():
    ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
    if ckpt_files:
        latest_ckpt = os.path.join(checkpoint_dir, sorted(ckpt_files)[-1])
        checkpoint = torch.load(latest_ckpt, map_location=device)
        netG.load_state_dict(checkpoint["netG"])
        netD.load_state_dict(checkpoint["netD"])
        optimizerG.load_state_dict(checkpoint["optimizerG"])
        optimizerD.load_state_dict(checkpoint["optimizerD"])
        scaler.load_state_dict(checkpoint["scaler"])
        print(f"Resuming from epoch {checkpoint['epoch']+1}")
        return checkpoint['epoch'] + 1
    return 0

start_epoch = load_checkpoint()

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)  # label smoothing
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 64        # depends on your dataset
nz = 100               # latent vector size
display_interval = 100
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1, 1] range
])

# Replace with your dataset
# dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator and Discriminator
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.main(x).view(-1)

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, Loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(device_type='cuda')  # updated AMP usage

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint functions
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

def load_checkpoint():
    ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
    if ckpt_files:
        latest_ckpt = os.path.join(checkpoint_dir, sorted(ckpt_files)[-1])
        checkpoint = torch.load(latest_ckpt, map_location=device)
        netG.load_state_dict(checkpoint["netG"])
        netD.load_state_dict(checkpoint["netD"])
        optimizerG.load_state_dict(checkpoint["optimizerG"])
        optimizerD.load_state_dict(checkpoint["optimizerD"])
        scaler.load_state_dict(checkpoint["scaler"])
        print(f"Resuming from epoch {checkpoint['epoch']+1}")
        return checkpoint['epoch'] + 1
    return 0

# -----------------------------
# Start fresh (ignore old checkpoints if desired)
# -----------------------------
start_epoch = load_checkpoint()  # will return 0 if no checkpoints exist

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)  # label smoothing
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        with torch.amp.autocast():
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.amp.autocast():
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 64        # adjust based on dataset
nz = 100               # latent vector size
display_interval = 100
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1]
])

# Example dataset (replace with yours)
# dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator and Discriminator
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.main(x).view(-1)

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, Loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()  # compatible with Kaggle PyTorch

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint functions
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

def load_checkpoint():
    ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
    if ckpt_files:
        latest_ckpt = os.path.join(checkpoint_dir, sorted(ckpt_files)[-1])
        checkpoint = torch.load(latest_ckpt, map_location=device)
        netG.load_state_dict(checkpoint["netG"])
        netD.load_state_dict(checkpoint["netD"])
        optimizerG.load_state_dict(checkpoint["optimizerG"])
        optimizerD.load_state_dict(checkpoint["optimizerD"])
        scaler.load_state_dict(checkpoint["scaler"])
        print(f"Resuming from epoch {checkpoint['epoch']+1}")
        return checkpoint['epoch'] + 1
    return 0

# Start fresh or resume if checkpoint exists
start_epoch = load_checkpoint()

# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)  # label smoothing
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os
import shutil

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 64
nz = 100  # latent vector size
display_interval = 100
checkpoint_dir = "checkpoints"

# -----------------------------
# Clear old checkpoints (start fresh)
# -----------------------------
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset (replace with your own)
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Example dataset
# dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# -----------------------------
# Discriminator
# -----------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.main(x).view(-1)

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()  # works on Kaggle

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint functions
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

# -----------------------------
# Training loop
# -----------------------------
start_epoch = 0  # fresh start

for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)  # label smoothing
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os
import shutil

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 128
nz = 100
display_interval = 100
checkpoint_dir = "checkpoints"

# -----------------------------
# Clear old checkpoints
# -----------------------------
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))  # RGB normalization
])

# Example dataset (replace with your own)
# dataset = torchvision.datasets.ImageFolder(root="./data", transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator (3-channel output)
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),  # RGB output
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# -----------------------------
# Discriminator (3-channel input)
# -----------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),  # RGB input
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.main(x).view(-1)

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, Loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler()  # AMP for Kaggle

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint function
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

# -----------------------------
# Training loop
# -----------------------------
start_epoch = 0

for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        with torch.cuda.amp.autocast():
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os
import shutil

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 128
nz = 100
display_interval = 100
checkpoint_dir = "checkpoints"

# -----------------------------
# Clear old checkpoints
# -----------------------------
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))  # RGB normalization
])

# Replace with your own dataset
# dataset = torchvision.datasets.ImageFolder(root="./data", transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# -----------------------------
# Discriminator (outputs one logit per image)
# -----------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),   # 128x128 -> 64x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), # 64x64 -> 32x32
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), # 32x32 -> 16x16
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1024, 4, 2, 1, bias=False),# 16x16 -> 8x8
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(1024, 1, 8, 1, 0, bias=False)  # 8x8 -> 1x1
        )

    def forward(self, x):
        return self.main(x).view(-1)  # [batch_size]

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, Loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler()  # updated AMP usage

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint function
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

# -----------------------------
# Training loop
# -----------------------------
start_epoch = 0

for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        with torch.amp.autocast():
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.amp.autocast():
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch: display generated images
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import os
import shutil

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
# -----------------------------
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0001
beta1 = 0.5
beta2 = 0.999
image_size = 128
nz = 100
display_interval = 100
checkpoint_dir = "checkpoints"

# -----------------------------
# Clear old checkpoints
# -----------------------------
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))  # RGB normalization
])

# Replace with your own dataset
# dataset = torchvision.datasets.ImageFolder(root="./data", transform=transform)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator
# -----------------------------
class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# -----------------------------
# Discriminator (outputs one logit per image)
# -----------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),   # 128x128 -> 64x64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), # 64x64 -> 32x32
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), # 32x32 -> 16x16
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1024, 4, 2, 1, bias=False),# 16x16 -> 8x8
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(1024, 1, 8, 1, 0, bias=False)  # 8x8 -> 1x1
        )

    def forward(self, x):
        return self.main(x).view(-1)  # [batch_size]

# -----------------------------
# Weight initialization
# -----------------------------
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = Generator(nz).to(device)
netD = Discriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

# -----------------------------
# Optimizers, Loss, AMP
# -----------------------------
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, beta2))
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, beta2))
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler()  # AMP usage

fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint function
# -----------------------------
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
        "scaler": scaler.state_dict()
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

# -----------------------------
# Training loop
# -----------------------------
start_epoch = 0

for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 0.9, device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        optimizerD.zero_grad(set_to_none=True)
        noise = torch.randn(b_size, nz, 1, 1, device=device)

        # Corrected autocast usage with device_type
        with torch.amp.autocast(device_type='cuda'):
            output_real = netD(real)
            lossD_real = criterion(output_real, label_real)
            fake = netG(noise)
            output_fake = netD(fake.detach())
            lossD_fake = criterion(output_fake, label_fake)
            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimizerD)

        # Train Generator
        optimizerG.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda'):
            output = netD(fake)
            lossG = criterion(output, label_real)
        scaler.scale(lossG).backward()
        scaler.step(optimizerG)
        scaler.update()

        if i % display_interval == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch {i}/{len(dataloader)} "
                  f"Loss_D: {lossD.item():.4f} Loss_G: {lossG.item():.4f}")

    # End of epoch: display generated images
    print(f"--- Epoch {epoch+1} completed ---")
    with torch.no_grad():
        fake_test = netG(fixed_noise).detach().cpu()
    grid = vutils.make_grid(fake_test, padding=2, normalize=True)
    plt.figure(figsize=(6,6))
    plt.axis("off")
    plt.title(f"Epoch {epoch+1} Output")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()
    plt.close()

    save_checkpoint(epoch)

print("Training complete. Final outputs are ready.")


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch
import os

# -----------------------------
# Hyperparameters
# -----------------------------
fixed_noise = torch.randn(64, 100, 1, 1, device=device)
display_interval = 5    # show output every 5 batches
num_epochs = 500        # increased from 5
gen_lr = 0.0004         # generator learning rate

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Checkpoint directory
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Training loop
# -----------------------------
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()
            plt.close()

    # --------- Save checkpoint at the end of the epoch ---------
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "netG_state_dict": netG.state_dict(),
        "netD_state_dict": netD.state_dict(),
        "optimizerG_state_dict": optimizerG.state_dict(),
        "optimizerD_state_dict": optimizerD.state_dict()
    }, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

print("Training complete.")


In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torch
import os

# -----------------------------
# Hyperparameters
# -----------------------------
fixed_noise = torch.randn(64, 100, 1, 1, device=device)
display_interval = 5    # show output every 5 batches
num_epochs = 500        # increased from 5
gen_lr = 0.0004         # generator learning rate

# Update optimizer if needed
optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))

# Checkpoint directory
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Training loop
# -----------------------------
for epoch in range(num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()
            plt.close()

    # --------- Save checkpoint at the end of the epoch ---------
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "netG_state_dict": netG.state_dict(),
        "netD_state_dict": netD.state_dict(),
        "optimizerG_state_dict": optimizerG.state_dict(),
        "optimizerD_state_dict": optimizerD.state_dict()
    }, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

print("Training complete.")

  

In [ ]:
import os
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
netG = netG.to(device)
netD = netD.to(device)

# -----------------------------
# Hyperparameters
# -----------------------------
fixed_noise = torch.randn(64, 100, 1, 1, device=device)
display_interval = 5      # show output every 5 batches
num_epochs = 500          # increase as needed
gen_lr = 0.0004           # generator learning rate

optimizerG = torch.optim.Adam(netG.parameters(), lr=gen_lr, betas=(0.5, 0.999))
optimizerD = torch.optim.Adam(netD.parameters(), lr=0.0001, betas=(0.5, 0.999))  # example

checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Load checkpoint if available
# -----------------------------
start_epoch = 0
ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
if ckpt_files:
    latest_ckpt = os.path.join(checkpoint_dir, sorted(ckpt_files)[-1])
    checkpoint = torch.load(latest_ckpt, map_location=device)
    netG.load_state_dict(checkpoint["netG_state_dict"])
    netD.load_state_dict(checkpoint["netD_state_dict"])
    optimizerG.load_state_dict(checkpoint["optimizerG_state_dict"])
    optimizerD.load_state_dict(checkpoint["optimizerD_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {start_epoch}")

# -----------------------------
# Training loop
# -----------------------------
for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # --------- Train Discriminator ---------
        netD.zero_grad()
        output_real = netD(real)
        lossD_real = criterion(output_real, label_real)

        noise = torch.randn(b_size, 100, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD_fake = criterion(output_fake, label_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --------- Train Generator ---------
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        # --------- Display intermediate results ---------
        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()
            plt.close()

    # --------- Save checkpoint at the end of the epoch ---------
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "netG_state_dict": netG.state_dict(),
        "netD_state_dict": netD.state_dict(),
        "optimizerG_state_dict": optimizerG.state_dict(),
        "optimizerD_state_dict": optimizerD.state_dict()
    }, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

print("Training complete.")



In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# -----------------------------
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Hyperparameters
image_size = 64
nz = 100
ngf = 64
ndf = 64
num_epochs = 100
batch_size = 64
lrG = 0.0004
lrD = 0.0004
beta1 = 0.5
display_interval = 5
checkpoint_dir = "./checkpoints"

os.makedirs(checkpoint_dir, exist_ok=True)

# -----------------------------
# Data loader (example: CIFAR10)
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = torchvision.datasets.CIFAR10(root='./data', download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# -----------------------------
# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

# -----------------------------
# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)

# -----------------------------
# Initialize models
netG = Generator().to(device)
netD = Discriminator().to(device)

# -----------------------------
# Loss and optimizers
criterion = nn.BCEWithLogitsLoss()
optimizerG = optim.Adam(netG.parameters(), lr=lrG, betas=(beta1, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=lrD, betas=(beta1, 0.999))

# Fixed noise for monitoring progress
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# -----------------------------
# Checkpoint functions
def save_checkpoint(epoch):
    torch.save({
        "epoch": epoch,
        "netG": netG.state_dict(),
        "netD": netD.state_dict(),
        "optimizerG": optimizerG.state_dict(),
        "optimizerD": optimizerD.state_dict(),
    }, os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth"))

def load_checkpoint():
    ckpt_files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
    if ckpt_files:
        latest_ckpt = os.path.join(checkpoint_dir, sorted(ckpt_files)[-1])
        checkpoint = torch.load(latest_ckpt, map_location=device)
        netG.load_state_dict(checkpoint["netG"])
        netD.load_state_dict(checkpoint["netD"])
        optimizerG.load_state_dict(checkpoint["optimizerG"])
        optimizerD.load_state_dict(checkpoint["optimizerD"])
        return checkpoint["epoch"] + 1
    return 0

start_epoch = load_checkpoint()

# -----------------------------
# Training loop
for epoch in range(start_epoch, num_epochs):
    for i, (real, _) in enumerate(dataloader):
        b_size = real.size(0)
        real = real.to(device)
        label_real = torch.full((b_size,), 1., device=device)
        label_fake = torch.full((b_size,), 0., device=device)

        # Train Discriminator
        netD.zero_grad()
        output_real = netD(real)
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)
        output_fake = netD(fake.detach())
        lossD = criterion(output_real, label_real) + criterion(output_fake, label_fake)
        lossD.backward()
        optimizerD.step()

        # Train Generator
        netG.zero_grad()
        output = netD(fake)
        lossG = criterion(output, label_real)
        lossG.backward()
        optimizerG.step()

        if i % display_interval == 0:
            print(f"Epoch {epoch+1} Batch {i}: Loss_D={lossD.item():.4f}, Loss_G={lossG.item():.4f}")
            with torch.no_grad():
                fake_test = netG(fixed_noise).detach().cpu()
            grid = vutils.make_grid(fake_test, padding=2, normalize=True)
            plt.figure(figsize=(6,6))
            plt.axis("off")
            plt.title(f"Epoch {epoch+1} Batch {i}")
            plt.imshow(grid.permute(1,2,0).numpy())
            plt.show()

    # Save checkpoint every epoch
    save_checkpoint(epoch)
